## Proposed Bus Routes - BTO

In [86]:
import folium
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import LineString, Point
from mrt_map import get_mrt_map

In [87]:
mrt_map = get_mrt_map()
mrt_map

/Users/krystal/Documents/GitHub/DSA4264/DSA4264/venv/lib/python3.10/site-packages/pyogrio/raw.py:198: RuntimeWarning: TrainStation_Jul2024/repaired_shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/krystal/Documents/GitHub/DSA4264/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()
/Users/krystal/Documents/GitHub/DSA4264/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geo

In [88]:
df_202407 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202407.csv")
df_202408 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202408.csv")
df_202409 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202409.csv")
combined_df = pd.concat([df_202407, df_202408, df_202409], ignore_index=True)

# Filter for DAY_TYPE == 'WEEKDAY' and TIME_PER_HOUR for peak hours [7, 8, 9, 10, 17, 18, 19, 20]
filtered_df = combined_df[(combined_df['DAY_TYPE'] == 'WEEKDAY') & 
                          (combined_df['TIME_PER_HOUR'].isin([7, 8, 9, 10, 17, 18, 19, 20]))]

# Create an empty summarized DataFrame
summarised_df = pd.DataFrame(columns=['PT_CODE', 'TOTAL_VOLUME'])

# Group by 'PT_CODE' and calculate the sum of tap-in and tap-out volumes
grouped = filtered_df.groupby('PT_CODE').agg(
    TOTAL_TAP_IN_VOLUME=('TOTAL_TAP_IN_VOLUME', 'sum'),
    TOTAL_TAP_OUT_VOLUME=('TOTAL_TAP_OUT_VOLUME', 'sum')
).reset_index()

# Create a new column for the total volume (sum of tap-in and tap-out volumes)
grouped['TOTAL_VOLUME'] = grouped['TOTAL_TAP_IN_VOLUME'] + grouped['TOTAL_TAP_OUT_VOLUME']

# Assign the grouped results to summarised_df
summarised_df['PT_CODE'] = grouped['PT_CODE']
summarised_df['TOTAL_VOLUME'] = grouped['TOTAL_VOLUME']
summarised_df = summarised_df.sort_values(by='TOTAL_VOLUME', ascending=False)
# summarised_df

trunkroutes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")
# trunkroutes
trunkroutes_grouped = trunkroutes.groupby('BusStopCode').first().reset_index()

# Merge summarised_df with trunkroutes based on PT_CODE == BusStopCode
merged_df = pd.merge(summarised_df, trunkroutes_grouped[['BusStopCode', 'Latitude', 'Longitude']], 
                     left_on='PT_CODE', right_on='BusStopCode', how='left')
merged_df = merged_df.drop(columns=['BusStopCode'])
merged_df.to_csv('location_popular_bus_Stops.csv', index=False)
merged_df

,PT_CODE,TOTAL_VOLUME,Latitude,Longitude
0,22009,4485567,1.339323,103.705457
1,46009,4185025,1.436946,103.785936
2,75009,3124114,1.354076,103.943391
3,52009,2189114,1.332021,103.846928
4,59009,1947889,1.428400,103.836097
...,...,...,...,...
5096,32059,15,1.409031,103.701208
5097,22579,14,NaN,NaN
5098,32069,8,1.411033,103.700381
5099,32049,4,1.404523,103.701870


In [89]:
# Create a function to scale the size of the marker based on TOTAL_VOLUME
def scale_marker_size(volume, min_size=5, max_size=15):
    volume_range = merged_df['TOTAL_VOLUME'].max() - merged_df['TOTAL_VOLUME'].min()
    if volume_range == 0:
        return min_size  # Avoid division by zero
    scaled_size = ((volume - merged_df['TOTAL_VOLUME'].min()) / volume_range) * (max_size - min_size) + min_size
    return scaled_size

## Add BTO Locations

In [90]:
## BTOs with over 1,000 units
taman_jurong_skyline = [1.3270289195127982, 103.72582148080623]
tanjong_rhu = [1.2996481809089269, 103.88025809169478]
teban_breeze = [1.3218378113117057, 103.74476401645141]
chencharu_hills = [1.419134809710128, 103.82527849062635]
marsiling_peak = [1.444461323034268, 103.77610683781455]
woodgrove_edge = [1.4290005301535544, 103.78480587301898]
#holland_vista= [1.3097277680430797, 103.79456384085621]


# Add a red circle marker for the bto locations
folium.CircleMarker(
    location=taman_jurong_skyline,
    radius=20, 
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Taman Jurong Skyline", max_width=100)
).add_to(mrt_map)


folium.CircleMarker(
    location=tanjong_rhu,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Tanjong Rhu Riverfront I & II", max_width=100)
).add_to(mrt_map)

folium.CircleMarker(
    location=teban_breeze,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Teban Breeze", max_width=100)
).add_to(mrt_map)

folium.CircleMarker(
    location=chencharu_hills,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Chencharu Hills", max_width=100)
).add_to(mrt_map)

folium.CircleMarker(
    location=marsiling_peak,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("Marsiling Peak I & II", max_width=100)
).add_to(mrt_map)

folium.CircleMarker(
    location=woodgrove_edge,
    radius=20,  
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.7,
    popup=folium.Popup("WoodGrove Edge", max_width=100)
).add_to(mrt_map)


# Display the map
mrt_map

## Plot bus routes for buses servicing each BTO 

In [91]:
bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")

nearest_stop_codes = {
    'taman_jurong_skyline': 21019,
    'tanjong_rhu': 90051,
    'teban_breeze': 20261,
    'chencharu_hills': 57069,
    'marsiling_peak': 46121,
    'woodgrove_edge': 46229
}

bto_colors = {
    'taman_jurong_skyline': 'blue',
    'tanjong_rhu': 'green',
    'teban_breeze': 'orange',
    'chencharu_hills': 'purple',
    'marsiling_peak': 'brown',
    'woodgrove_edge': 'pink'
}

filtered_routes = {}

for bto, stop_code in nearest_stop_codes.items():
    # Find the bus services stopping at the nearest bus stop
    bto_services = bus_routes[bus_routes['BusStopCode'] == stop_code]['ServiceNo'].unique()
    
    # Filter the bus_routes DataFrame to include only those services
    filtered_routes[bto] = bus_routes[bus_routes['ServiceNo'].isin(bto_services)]

# Plot the routes for each filtered service, and add only the markers for the stops they visit
for bto, routes in filtered_routes.items():
    color = bto_colors[bto]  # Get the color for the current BTO location
    for service_no in routes['ServiceNo'].unique():
        # Extract route points for each service
        route_points = routes[routes['ServiceNo'] == service_no][['Latitude', 'Longitude']].values
        
        # Draw the route on the map
        folium.PolyLine(
            locations=route_points,
            color=color,  # Use the color corresponding to the BTO location
            weight=3,
            opacity=0.6,
            popup=f"Service {service_no} - {bto}"
        ).add_to(mrt_map)
        
        # Add markers for each stop along this route
        for idx, row in routes[routes['ServiceNo'] == service_no].iterrows():
            folium.CircleMarker(
                location=[row['Latitude'], row['Longitude']],
                radius=5,
                color=color,
                fill=True,
                fill_color=color,
                fill_opacity=0.7,
                popup=f"Bus Stop {row['BusStopCode']} - Service {service_no}"
            ).add_to(mrt_map)


# Display the map with filtered routes and stops
mrt_map
